# ProBat Insight - Google Colab Training Notebook

This notebook:
1. Clones the **Probat-Insight** repo
2. Installs all ML dependencies (works on Python 3.10 / 3.11 in Colab)
3. Trains the batting quality model
4. Downloads the saved model files back to your PC

> **After running**, copy the downloaded files into `backend/ml/saved_model/` in your local project.

## 1  Clone Repository

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/Anjalee-jay/Probat-Insight.git'
REPO_DIR = '/content/Probat-Insight'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned - pulling latest changes')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(os.path.join(REPO_DIR, 'backend'))
print('Working directory:', os.getcwd())

## 2  Install Dependencies

In [ ]:
import subprocess
import sys

# Pin versions to match requirements.txt
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'mediapipe==0.10.18',
    'opencv-python-headless==4.10.0.84',
    'numpy==1.26.4',
    'scikit-learn==1.5.2',
    'pandas==2.2.3',
    'joblib==1.4.2',
    'Pillow==10.4.0',
], check=True)

print('All ML dependencies installed.')

## 3  Train the Batting Quality Model

In [ ]:
import os
import subprocess
import sys

sys.path.insert(0, '/content/Probat-Insight/backend')

# Run training - saves batting_model.joblib + label_cols.json + training_report.txt
subprocess.run(
    [sys.executable, '-m', 'ml.train_model'],
    cwd='/content/Probat-Insight/backend',
    check=True
)

print('\n--- Saved artefacts ---')
for fname in sorted(os.listdir('ml/saved_model')):
    print(fname)

## 4  View Training Report

In [ ]:
with open('ml/saved_model/training_report.txt') as f:
    print(f.read())

## 5  Download Model Files to Your PC

In [ ]:
from google.colab import files  # type: ignore[import]

files.download('ml/saved_model/batting_model.joblib')
files.download('ml/saved_model/label_cols.json')
files.download('ml/saved_model/training_report.txt')

print('Downloaded! Copy these 3 files into: backend/ml/saved_model/')

## 6  (Optional) Test Inference on a Sample Image

Upload a cricket batting image to test the full pipeline.

In [ ]:
from google.colab import files as colab_files  # type: ignore[import]

uploaded = colab_files.upload()  # pick a .jpg / .png from your PC
image_name = list(uploaded.keys())[0]
print(f'Uploaded: {image_name}')

In [ ]:
import sys

import pandas as pd
from IPython.display import display  # type: ignore[import]

sys.path.insert(0, '/content/Probat-Insight/backend')

from ml.predictor import BattingPredictor  # type: ignore[import]

predictor = BattingPredictor()

with open(image_name, 'rb') as f:
    image_bytes = f.read()

result = predictor.analyse_image(image_bytes)

if result.get('error'):
    print('Error:', result['error'])
else:
    scores = result.get('scores', {})
    df = pd.DataFrame([
        {'Category': k.replace('_', ' ').title(), 'Score': f"{v:.1f} / 100"}
        for k, v in scores.items()
    ])
    display(df)
    print('\nOverall:', result.get('overall_score'), '/ 100')
    print('Grade  :', result.get('grade'))